Congrats! You just graduated UVA's BSDS program and got a job working at a movie studio in Hollywood. 

Your boss is the head of the studio and wants to know if they can gain a competitive advantage by predicting new movies that might get high imdb scores (movie rating). 

You would like to be able to explain the model to mere mortals but need a fairly robust and flexible approach so you've chosen to use decision trees to get started. 

In doing so, similar to  great data scientists of the past you remembered the excellent education provided to you at UVA in a undergrad data science course and have outline 20ish steps that will need to be undertaken to complete this task. As always, you will need to make sure to #comment your work heavily. 

 Footnotes: 
-	You can add or combine steps if needed
-	Also, remember to try several methods during evaluation and always be mindful of how the model will be used in practice.
- Make sure all your variables are the correct type (factor, character,numeric, etc.)

In [ ]:
# Imports
import os
os.environ["PATH"] += os.pathsep + "/opt/homebrew/bin"


import pandas as pd
import numpy as np
pd.options.mode.chained_assignment = None  # default='warn'
import matplotlib.pyplot as plt
import graphviz

from sklearn.model_selection import train_test_split,GridSearchCV,RepeatedStratifiedKFold
from sklearn import metrics
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier, export_graphviz 

In [ ]:
#1. Load the data
#Sometimes need to set the working directory back out of a folder that we create a file in

#import os
#os.listdir()
#print(os.getcwd())
#os.chdir('c:\\Users\\Brian Wright\\Documents\\3001Python\\DS-3001')

movie_metadata=pd.read_csv("../data/movie_metadata.csv")

2 Ensure all the variables are classified correctly including the target variable and collapse factor variables as needed.

3 Check for missing variables and correct as needed. Once you've completed the cleaning again create a function that will do this for you in the future. In the submission, include only the function and the function call.

4 Guess what, you don't need to scale the data, because DTs don't require this to be done, they make local greedy decisions...keeps getting easier, go to the next step.

5 Determine the baserate or prevalence for the classifier, what does this number mean?

6 Split your data into test, tune, and train. (80/10/10)

7 Create the kfold object for cross validation.

8 Create the scoring metric you will use to evaluate your model and the max depth hyperparameter (grid search) 

9 Build the classifier object 

10 Use the kfold object and the scoring metric to find the best hyperparameter value for max depth via the grid search method.

11 Fit the model to the training data.

12 What is the best depth value?

13 Print out the model

14 View the results, comment on how the model performed using the metrics you selected.

15 Which variables appear to be contributing the most (variable importance) 

16 Use the predict method on the tune data and print out the results.

17 How does the model perform on the tune data?

18 Print out the confusion matrix for the tune data, what does it tell you about the model?

19 What are the top 3 movies based on the tune set? Which variables are most important in predicting the top 3 movies?

20 Use a different hyperparameter for the grid search function and go through the process above again using the tune set. 

21 Did the model improve with the new hyperparameter search?

22 Using the better model, predict the test data and print out the results.

23 Summarize what you learned along the way and make recommendations to your boss on how this could be used moving forward, being careful not to over promise.

## Steps 1-4: Cleaning data

In [ ]:
def preprocess_movies_data(movie_metadata):
    # Keeping columns that are needed
    cols_to_keep = ['color', 'num_critic_for_reviews', 'duration', 'gross', 
                    'num_voted_users', 'country', 'language', 'content_rating', 
                    'budget', 'title_year', 'aspect_ratio', 'imdb_score']
    movies_data = movie_metadata[cols_to_keep].copy()

    # Recode country to USA vs Other
    movies_data['country'] = movies_data['country'].apply(
        lambda x: 'USA' if x == 'USA' else 'Other'
    )

    # Recode language to English vs Other
    movies_data['language'] = movies_data['language'].apply(
        lambda x: 'English' if x == 'English' else 'Other'
    )

    # Recode content_rating to standard ratings vs Other
    valid_ratings = ['G', 'PG', 'PG-13', 'R']
    movies_data['content_rating'] = movies_data['content_rating'].apply(lambda x: x if x in valid_ratings else 'Other')

    # Convert categorical variables
    label_encoder = LabelEncoder()
    for col in ['color', 'country', 'language', 'content_rating']:
        movies_data[col] = label_encoder.fit_transform(movies_data[col])

    # Drop rows with missing values
    movies_data = movies_data.dropna()

    return movies_data

cleaned_movies_data = preprocess_movies_data(movie_metadata)
cleaned_movies_data


## Steps 5-13: Building out the model

In [ ]:
# Encoding IMDB score as 1 if >7, 0 if not
cleaned_movies_data['imdb_score'] = cleaned_movies_data['imdb_score'].apply(lambda x: 1 if x > 7 else 0)

high_score_ratio = cleaned_movies_data['imdb_score'].mean()
print(high_score_ratio)


30% is a good level for our prevalnce of good movies

In [ ]:
# Splitting into our independent and dependent variables
X= cleaned_movies_data.drop(columns='imdb_score')
y= cleaned_movies_data.imdb_score

# Train/tune/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8, stratify= y, random_state=13)
X_tune, X_test, y_tune, y_test = train_test_split(X_test,y_test,  train_size = 0.50,stratify= y_test, random_state=72)

# Creating k-fold object for validiation
kf = RepeatedStratifiedKFold(n_splits=10,n_repeats =5, random_state=42)

# Selecting scoring metrics
scoring = ['roc_auc','f1','r2']

# Building classifier object
param={"max_depth" : [1,2,3,4,5,6,7,8,9,10,11]}
cl= DecisionTreeClassifier(random_state=500)
search = GridSearchCV(cl, param, scoring=scoring, n_jobs=-1, cv=kf,refit='roc_auc')

# Fitting to training data, finding best depth
model = search.fit(X_train, y_train)
best = model.best_estimator_
print(best)


A depth of 6 is best given our data

## Steps 14-19: Assessing model performance

In [ ]:
#Scores: 
auc = model.cv_results_['mean_test_roc_auc']
f1 = model.cv_results_['mean_test_f1']
r2 = model.cv_results_['mean_test_r2']

SD_auc = model.cv_results_['std_test_roc_auc']
SD_f1 = model.cv_results_['std_test_f1']
SD_r2= model.cv_results_['std_test_r2']

#Parameter:
depth= np.unique(model.cv_results_['param_max_depth']).data

#Build DataFrame:
final_model = pd.DataFrame(list(zip(depth, auc, f1, r2,SD_auc,SD_f1,SD_r2)),
               columns =['depth','auc','f1','r2','aucSD','f1SD','r2SD'])

# Look at dataframe with all necessary statistics
final_model.style.hide(axis='index')


We can see here that auc is highest at a depth of 6. f1 score also seems to perform best around 6

In [ ]:
# Checking variable importance:
varimp=pd.DataFrame(best.feature_importances_,index = X.columns,columns=['importance']).sort_values('importance', ascending=False)
print(varimp)

Number of voted users is by far the most important variable, followed by budget and language(English/other)

In [ ]:
# Checking accuracy: 
from sklearn.metrics import accuracy_score
 
y_pred = model.predict(X_tune)
accuracy = accuracy_score(y_tune, y_pred)
print("Accuracy:", accuracy)

Our model has an accuracy of around 80%

In [ ]:
print(ConfusionMatrixDisplay.from_estimator(best,X_tune,y_tune, display_labels = ['ave/poor','strong'], colorbar=False))

The confusion matrix tells us that the model does well at predicting a movie will be average/poor when it is but is not as good at classifying strong movies as strong.

##

In [ ]:
# Predict probabilities on the tune set
y_probs = best.predict_proba(X_tune)[:, 1]

# Attach probabilities to the original tune set
tune_with_probs = X_tune.copy()
tune_with_probs['predicted_prob'] = y_probs

top_3 = tune_with_probs.sort_values(by='predicted_prob', ascending=False).head(3)
print(top_3)


The movies with the highest likelihood of being highly rated on IMDB were "The Matrix: Reloaded", "Casino Royale", and "Die Hard". The number of reviews, budget, and duration were most important.

## Steps 20-23: Reproducing with new parameter

In [ ]:
# Selecting min_samples_split as new param
param_1={"min_samples_split":[5,10,15,20,25],}
search_2 = GridSearchCV(cl, param_1, scoring=scoring, n_jobs=-1, cv=kf,refit='roc_auc')

model_2 = search_2.fit(X_train, y_train)
#Scores: 
auc_2 = model_2.cv_results_['mean_test_roc_auc']
f1_2 = model_2.cv_results_['mean_test_f1']
r2_2 = model_2.cv_results_['mean_test_r2']

SD_auc_2 = model_2.cv_results_['std_test_roc_auc']
SD_f1_2 = model_2.cv_results_['std_test_f1']
SD_r2_2 = model_2.cv_results_['std_test_r2']

#Parameter:
min_samples = model_2.cv_results_['param_min_samples_split'].data

#Build DataFrame:
final_model = pd.DataFrame(list(zip(min_samples, auc_2, f1_2, r2_2,SD_auc_2,SD_f1_2,SD_r2_2)),
               columns =['depth','auc','f1','r2','aucSD','f1SD','r2SD'])

final_model.style.hide(axis='index')


The other model performed slightly better based on auc.

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score
# Predicting test data, printing results with first model
y_pred = best.predict(X_test)          
y_proba = best.predict_proba(X_test)[:, 1] 

print("Classification Report:\n", classification_report(y_test, y_pred))
print("Test ROC AUC:", roc_auc_score(y_test, y_proba))


### Summarize findings: 
Based off of this project, it is clear that not all variables contribute equally to success of movies. The most important predictor is number of reviews, which cannot be easily tailored to in the production process. It also seems that American movies with bigger budgets tend to do better than other movies based off of IMDB score. Some limitations I found were variables such as leading actor and director likely had an impact, but they cannot easily be considered with a decision tree. I also chose to not include facebook likes as I felt it wouldn't apply equal importance across movies, but that may have been an important factor. Going forward, I would reccomend generating buzz around movies and having audience involvement. Hosting events such as large pre-screenings for the public may lead to more reviews, which generally leads to better reviews. However, there are so many factors at play that there is no easily identifiable solution to creating a highly rated movie.

Note: I outlined all in a seperate file and piped them together here